# PCP search — live Supabase monitor

CPU runtime is enough. Add the same Supabase Secrets, then run the cells whenever you want a fresh view. This reads the two worker progress artifacts and can pull all completed/error rollout rows with pagination. It launches no experiments.


In [ ]:
import sys, subprocess, os, json
subprocess.run([sys.executable,'-m','pip','install','-q','supabase','pandas'],check=True)
from google.colab import userdata
from supabase import create_client
import pandas as pd
client=create_client(userdata.get('SUPABASE_URL'),userdata.get('SUPABASE_SERVICE_KEY'))
EXPERIMENT='smolvla-pcp-a100-jeff-settings-stock-v6'


In [ ]:
reports=[]
for current_experiment,worker in [(EXPERIMENT,0),(EXPERIMENT,1),('smolvla-pcp-full-proposal-jeff-settings-v5',0)]:
    key=f'smolvla_pcp_sweep/{current_experiment}/worker_{worker}_latest.json'
    try:
        report=json.loads(client.storage.from_('artifacts').download(key));reports.append(report)
        print({k:v for k,v in report.items() if k!='leaderboard'})
        display(pd.DataFrame(report['leaderboard']).head(20))
    except Exception as error:
        if not any(t in str(error).lower() for t in ('404','not found','does not exist')):raise
        print(f'{current_experiment} worker {worker}: first progress artifact is not available yet.')


In [ ]:
# Optional: pull raw episodes, including errors. Completed reruns upsert stable IDs.
PULL_RAW_EPISODES=False
if PULL_RAW_EPISODES:
    rows=[]
    for start in range(0,1000000,1000):
        batch=(client.table('rollouts').select('rollout_id,method,suite,task_idx,episode_idx,behavior_seed_index,success,status,error_msg,n_steps,elapsed_s,ms_candidate_u')
               .eq('experiment',EXPERIMENT).order('rollout_id').range(start,start+999).execute().data or [])
        rows.extend(batch)
        if len(batch)<1000:break
    episodes=pd.DataFrame(rows)
    print('Logged rows:',len(episodes))
    display(episodes.head())
    episodes.to_json('/content/pcp_search_episodes.jsonl',orient='records',lines=True)
